# 04 - Data Quality Checks

## Purpose

Run practical data quality checks before broad business consumption.

Checks include not null, duplicate, accepted values, range, referential integrity, row count reconciliation, and freshness.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_timestamp = datetime.now(timezone.utc).isoformat()
results = []


def add_result(table_name, rule_name, status, failed_count, details):
    results.append((table_name, rule_name, status, int(failed_count), details, run_timestamp))


def check_not_null(table_name, columns):
    df = spark.table(table_name)
    for column_name in columns:
        failed = df.filter(F.col(column_name).isNull()).count()
        add_result(table_name, f"not_null:{column_name}", "PASS" if failed == 0 else "FAIL", failed, f"Column {column_name} should not be null")


def check_duplicates(table_name, key_columns):
    duplicate_count = spark.table(table_name).groupBy(*key_columns).count().filter(F.col("count") > 1).count()
    add_result(table_name, "duplicate_key", "PASS" if duplicate_count == 0 else "FAIL", duplicate_count, ",".join(key_columns))


def check_accepted_values(table_name, column_name, values):
    failed = spark.table(table_name).filter(~F.col(column_name).isin(values)).count()
    add_result(table_name, f"accepted_values:{column_name}", "PASS" if failed == 0 else "FAIL", failed, f"Accepted values: {values}")


def check_range(table_name, column_name, min_value=None, max_value=None):
    condition = F.lit(False)
    if min_value is not None:
        condition = condition | (F.col(column_name) < F.lit(min_value))
    if max_value is not None:
        condition = condition | (F.col(column_name) > F.lit(max_value))
    failed = spark.table(table_name).filter(condition).count()
    add_result(table_name, f"range:{column_name}", "PASS" if failed == 0 else "FAIL", failed, f"Range {min_value} to {max_value}")


def check_referential_integrity(child_table, child_column, parent_table, parent_column):
    failed = spark.table(child_table).select(child_column).distinct().alias("c").join(spark.table(parent_table).select(parent_column).distinct().alias("p"), F.col(f"c.{child_column}") == F.col(f"p.{parent_column}"), "left_anti").count()
    add_result(child_table, f"referential_integrity:{child_column}", "PASS" if failed == 0 else "FAIL", failed, f"Must exist in {parent_table}.{parent_column}")

In [ ]:
check_not_null("silver_customers", ["customer_id", "customer_status", "customer_segment"])
check_duplicates("silver_customers", ["customer_id"])
check_accepted_values("silver_customers", "customer_status", ["Active", "Inactive"])

check_not_null("silver_accounts", ["account_id", "customer_id", "product_id", "branch_id", "account_status"])
check_duplicates("silver_accounts", ["account_id"])
check_accepted_values("silver_accounts", "account_status", ["Open", "Closed"])
check_range("silver_accounts", "current_balance", -1000000, 10000000)
check_referential_integrity("silver_accounts", "customer_id", "silver_customers", "customer_id")
check_referential_integrity("silver_accounts", "product_id", "silver_products", "product_id")

check_not_null("silver_transactions", ["transaction_id", "account_id", "transaction_timestamp", "transaction_amount", "transaction_status"])
check_duplicates("silver_transactions", ["transaction_id"])
check_accepted_values("silver_transactions", "transaction_status", ["Posted", "Rejected", "Pending"])
check_referential_integrity("silver_transactions", "account_id", "silver_accounts", "account_id")
check_referential_integrity("silver_transactions", "branch_id", "silver_branches", "branch_id")

In [ ]:
silver_transaction_count = spark.table("silver_transactions").count()
gold_fact_count = spark.table("fact_transaction").count()
failed_reconciliation = 0 if silver_transaction_count == gold_fact_count else abs(silver_transaction_count - gold_fact_count)
add_result("fact_transaction", "row_count_reconciliation", "PASS" if failed_reconciliation == 0 else "FAIL", failed_reconciliation, f"Silver={silver_transaction_count}, Gold={gold_fact_count}")

max_ingestion_ts = spark.table("silver_transactions").agg(F.max("_ingestion_timestamp").alias("max_ts")).collect()[0]["max_ts"]
add_result("silver_transactions", "freshness:_ingestion_timestamp", "PASS" if max_ingestion_ts is not None else "FAIL", 0 if max_ingestion_ts is not None else 1, f"Max ingestion timestamp: {max_ingestion_ts}")

In [ ]:
dq_report = spark.createDataFrame(results, ["table_name", "rule_name", "status", "failed_count", "details", "run_timestamp"])
display(dq_report.orderBy("status", "table_name", "rule_name"))

fail_count = dq_report.filter(F.col("status") == "FAIL").count()
print("# Data Quality Report")
print(f"Run timestamp: {run_timestamp}")
print(f"Total rules: {dq_report.count()}")
print(f"Failed rules: {fail_count}")

if fail_count > 0:
    raise ValueError("Data quality checks failed. Review the dq_report output before promoting Gold data.")
else:
    print("All data quality checks passed.")